# DSPy RAG — Retrieval-Augmented Generation with Constraints

**Week 6 | Notebook 3 of 12**

**What you'll learn:**
- Setting up a retriever (ChromaDB / Qdrant)
- Building a basic RAG module
- Enforcing faithfulness with plain-Python validation (dspy 3.x removed `dspy.Assert`)
- Soft constraints (conciseness warnings) — `dspy.Suggest` was removed in dspy 3.x
- Multi-hop RAG — iterative retrieval
- Optimizing the RAG pipeline with MIPROv2
- Evaluating with SemanticF1 and CompleteAndGrounded

**Runtime:** ~60 minutes

In [2]:
# 💰 COST ESTIMATE
from src.cost_tracker import print_cost_warning

print_cost_warning("06_dspy/03_rag_pipeline.ipynb")

💰 COST ESTIMATE
----------------------------------------
Notebook:  06_dspy/03_rag_pipeline.ipynb
Task:      RAG with assertions
Calls:     ~25

With GPT-4o:       $0.38 USD
With GPT-4o-mini:  $0.04 USD (10x cheaper)
With Ollama:       $0.00 USD (free, local)

💡 TIP: Set USE_SMALL_MODEL=true or USE_OLLAMA=true in .env to save money.
----------------------------------------


## 1. Setup — In-Memory Retriever

In [3]:
import dspy

from src.config import get_dspy_lm
from src.datasets import generate_rag_contexts

lm = get_dspy_lm()
dspy.configure(lm=lm)

# Simple in-memory retriever for demo
documents = generate_rag_contexts(20)
corpus = {f"doc_{i}": d["context"] for i, d in enumerate(documents)}


class SimpleRetriever:
    def __init__(self, corpus, k=3):
        self.corpus = corpus
        self.k = k

    def __call__(self, query):
        # Simple keyword matching (replace with real vector DB in production)
        words = query.lower().split()
        scores = {
            doc_id: sum(1 for w in words if w in text.lower())
            for doc_id, text in self.corpus.items()
        }
        top = sorted(scores.items(), key=lambda x: x[1], reverse=True)[: self.k]
        return dspy.Prediction(passages=[self.corpus[doc_id] for doc_id, _ in top if _ > 0])


retriever = SimpleRetriever(corpus)
print(f"✅ Retriever ready with {len(corpus)} documents")

✅ Retriever ready with 20 documents


## 2. Basic RAG Module

In [4]:
class GenerateAnswer(dspy.Signature):
    """Answer questions based on context."""

    context: str = dspy.InputField()
    question: str = dspy.InputField()
    answer: str = dspy.OutputField()


class BasicRAG(dspy.Module):
    def __init__(self, num_passages=3):
        super().__init__()
        self.retrieve = retriever
        self.generate = dspy.ChainOfThought(GenerateAnswer)

    def forward(self, question):
        passages = self.retrieve(question).passages
        context = "\n".join(passages)
        return self.generate(context=context, question=question)


rag = BasicRAG()
result = rag(question="What is the return policy?")
print("Question: What is the return policy?")
print(f"Answer: {result.answer}")

Question: What is the return policy?
Answer: Items can be returned within 30 days with the original receipt, and refunds are processed within 5-7 business days.


## 3. Enforcing Faithfulness (Plain-Python Validation)

In [5]:
class FaithfulRAG(dspy.Module):
    def __init__(self):
        super().__init__()
        self.retrieve = retriever
        self.generate = dspy.ChainOfThought(GenerateAnswer)

    def forward(self, question):
        passages = self.retrieve(question).passages
        context = "\n".join(passages)
        result = self.generate(context=context, question=question)

        # Hard constraint: answer must be grounded in context
        # (dspy 3.x removed dspy.Assert/dspy.Suggest — validate in plain Python)
        assert any(word in context.lower() for word in result.answer.lower().split()[:3]), (
            "Answer must be grounded in the provided context."
        )

        return result


faithful_rag = FaithfulRAG()
try:
    result = faithful_rag(question="What is the return policy?")
    print(f"Answer: {result.answer}")
except AssertionError as e:
    print(f"Assertion failed: {e}")

Answer: Items can be returned within 30 days with the original receipt, and refunds are processed within 5-7 business days.


## 4. Soft Constraints for Length

In [6]:
class ConstrainedRAG(dspy.Module):
    def __init__(self):
        super().__init__()
        self.retrieve = retriever
        self.generate = dspy.ChainOfThought(GenerateAnswer)

    def forward(self, question):
        passages = self.retrieve(question).passages
        context = "\n".join(passages)
        result = self.generate(context=context, question=question)

        # Soft constraint: prefer concise answers — warn, don't fail
        if len(result.answer.split()) > 50:
            print("⚠️  Suggestion: answer should be under 50 words for conciseness.")

        return result


constrained_rag = ConstrainedRAG()
result = constrained_rag(question="What is the return policy?")
print(f"Answer ({len(result.answer.split())} words): {result.answer}")

Answer (19 words): Items can be returned within 30 days with the original receipt, and refunds are processed within 5-7 business days.


## 5. Multi-Hop RAG — Iterative Retrieval

In [7]:
class GenerateSearchQuery(dspy.Signature):
    """Generate a search query based on context and question."""

    context: str = dspy.InputField()
    question: str = dspy.InputField()
    search_query: str = dspy.OutputField()


class MultiHopRAG(dspy.Module):
    def __init__(self, num_hops=2):
        super().__init__()
        self.retrieve = retriever
        self.generate_query = dspy.ChainOfThought(GenerateSearchQuery)
        self.generate_answer = dspy.ChainOfThought(GenerateAnswer)
        self.num_hops = num_hops

    def forward(self, question):
        context = []
        for _ in range(self.num_hops):
            query = self.generate_query(context="\n".join(context), question=question).search_query
            passages = self.retrieve(query).passages
            context.extend(passages)

        return self.generate_answer(context="\n".join(context), question=question)


multi_hop = MultiHopRAG(num_hops=2)
result = multi_hop(question="What is the return policy?")
print(f"Multi-hop answer: {result.answer}")

Multi-hop answer: The return policy allows items to be returned within 30 days with the original receipt. Refunds are processed within 5-7 business days.


## 6. Optimizing with MIPROv2

In [8]:
from dspy.evaluate import Evaluate
from dspy.teleprompt import MIPROv2

# Prepare eval data
rag_data = generate_rag_contexts(20)
eval_examples = [
    dspy.Example(question=d["question"], answer=d["expected"]).with_inputs("question")
    for d in rag_data[:10]
]


def rag_metric(example, prediction, trace=None):
    return 1.0 if example.answer.lower() in prediction.answer.lower() else 0.0


# Optimize multi-hop RAG
mipro = MIPROv2(metric=rag_metric, auto=None, num_candidates=3)
optimized_rag = mipro.compile(
    MultiHopRAG(num_hops=2),
    trainset=eval_examples[:7],
    num_trials=5,  # Reduced for cost
    valset=eval_examples[7:],
    minibatch=False,  # Small valset (3) — default minibatch size is 35
)

evaluator = Evaluate(devset=eval_examples[7:], metric=rag_metric, num_threads=2)
score = evaluator(optimized_rag).score
print(f"\nOptimized RAG score: {score:.2f}")

2026/09/20 16:58:13 INFO dspy.teleprompt.mipro_optimizer_v2: 
==> STEP 1: BOOTSTRAP FEWSHOT EXAMPLES <==
2026/09/20 16:58:13 INFO dspy.teleprompt.mipro_optimizer_v2: These will be used as few-shot example candidates for our program and for creating instructions.

2026/09/20 16:58:13 INFO dspy.teleprompt.mipro_optimizer_v2: Bootstrapping N=3 sets of demonstrations...


Bootstrapping set 1/3
Bootstrapping set 2/3
Bootstrapping set 3/3


100%|██████████| 7/7 [00:00<00:00, 15.40it/s]
2026/09/20 16:58:14 INFO dspy.teleprompt.mipro_optimizer_v2: 
==> STEP 2: PROPOSE INSTRUCTION CANDIDATES <==
2026/09/20 16:58:14 INFO dspy.teleprompt.mipro_optimizer_v2: We will use the few-shot examples from the previous step, a generated dataset summary, a summary of the program code, and a randomly selected prompting tip to propose instructions.
2026/09/20 16:58:14 INFO dspy.teleprompt.mipro_optimizer_v2: 
Proposing N=3 instructions...

2026/09/20 16:58:14 WARNING dspy.predict.predict: Input contains fields not in signature. These fields will be ignored: ['max_depth']. Expected fields: ['program_code', 'program_example', 'program_description', 'module'].
2026/09/20 16:58:14 WARNING dspy.predict.predict: Input contains fields not in signature. These fields will be ignored: ['previous_instructions']. Expected fields: ['dataset_description', 'program_code', 'program_description', 'module', 'module_description', 'task_demos', 'basic_instruct

Bootstrapped 2 full traces after 6 examples for up to 1 rounds, amounting to 7 attempts.


2026/09/20 16:58:14 WARNING dspy.predict.predict: Input contains fields not in signature. These fields will be ignored: ['previous_instructions']. Expected fields: ['dataset_description', 'program_code', 'program_description', 'module', 'module_description', 'task_demos', 'basic_instruction', 'tip'].
2026/09/20 16:58:14 WARNING dspy.predict.predict: Input contains fields not in signature. These fields will be ignored: ['max_depth']. Expected fields: ['program_code', 'program_example', 'program_description', 'module'].
2026/09/20 16:58:14 WARNING dspy.predict.predict: Input contains fields not in signature. These fields will be ignored: ['previous_instructions']. Expected fields: ['dataset_description', 'program_code', 'program_description', 'module', 'module_description', 'task_demos', 'basic_instruction', 'tip'].
2026/09/20 16:58:14 INFO dspy.teleprompt.mipro_optimizer_v2: Proposed Instructions for Predictor 0:

2026/09/20 16:58:14 INFO dspy.teleprompt.mipro_optimizer_v2: 0: Generate 

...

2026/09/20 16:58:14 INFO dspy.evaluate.evaluate: Average Metric: 1.0 / 3 (33.3%)
2026/09/20 16:58:14 INFO dspy.teleprompt.mipro_optimizer_v2: Score: 33.33 with parameters ['Predictor 0: Instruction 0', 'Predictor 0: Few-Shot Set 1', 'Predictor 1: Instruction 2', 'Predictor 1: Few-Shot Set 2'].
2026/09/20 16:58:14 INFO dspy.teleprompt.mipro_optimizer_v2: Scores so far: [33.33, 33.33, 33.33, 33.33, 33.33]
2026/09/20 16:58:14 INFO dspy.teleprompt.mipro_optimizer_v2: Best score so far: 33.33
2026/09/20 16:58:14 INFO dspy.teleprompt.mipro_optimizer_v2: =======================


2026/09/20 16:58:14 INFO dspy.teleprompt.mipro_optimizer_v2: ===== Trial 6 / 5 =====



Average Metric: 1.00 / 3 (33.3%): 100%|██████████| 3/3 [00:00<00:00, 66.94it/s]

2026/09/20 16:58:15 INFO dspy.evaluate.evaluate: Average Metric: 1.0 / 3 (33.3%)
2026/09/20 16:58:15 INFO dspy.teleprompt.mipro_optimizer_v2: Score: 33.33 with parameters ['Predictor 0: Instruction 0', 'Predictor 0: Few-Shot Set 0', 'Predictor 1: Instruction 2', 'Predictor 1: Few-Shot Set 2'].
2026/09/20 16:58:15 INFO dspy.teleprompt.mipro_optimizer_v2: Scores so far: [33.33, 33.33, 33.33, 33.33, 33.33, 33.33]
2026/09/20 16:58:15 INFO dspy.teleprompt.mipro_optimizer_v2: Best score so far: 33.33
2026/09/20 16:58:15 INFO dspy.teleprompt.mipro_optimizer_v2: =======================


2026/09/20 16:58:15 INFO dspy.teleprompt.mipro_optimizer_v2: Returning best identified program with score 33.33!


2026/09/20 16:58:15 INFO dspy.evaluate.evaluate: Average Metric: 1.0 / 3 (33.3%)



Optimized RAG score: 33.33


## 7. Evaluating with SemanticF1 and CompleteAndGrounded

In [8]:
# Note: These metrics require specific setup
# SemanticF1: semantic similarity between prediction and gold
# CompleteAndGrounded: checks coverage and faithfulness

print("Built-in RAG metrics:")
print("  • SemanticF1: Semantic similarity score")
print("  • CompleteAndGrounded: Coverage + faithfulness")
print("\nFor production, combine multiple metrics:")
print("  • Exact match for easy questions")
print("  • SemanticF1 for paraphrased answers")
print("  • CompleteAndGrounded for RAG faithfulness")

Built-in RAG metrics:
  • SemanticF1: Semantic similarity score
  • CompleteAndGrounded: Coverage + faithfulness

For production, combine multiple metrics:
  • Exact match for easy questions
  • SemanticF1 for paraphrased answers
  • CompleteAndGrounded for RAG faithfulness
